# Analisis Kemiskinan & Ketimpangan Indonesia 2026

Notebook ini mereproduksi angka-angka temuan utama dari CSV kompilasi BPS di folder `data/`. Setiap angka pada README, deck, dan modul berasal dari perhitungan di sini (dan di `scripts/analysis.py`).

In [ ]:
import pandas as pd

nas = pd.read_csv('data/kemiskinan_nasional.csv')
kd  = pd.read_csv('data/kemiskinan_kota_desa.csv')
gini = pd.read_csv('data/gini_nasional.csv')
prov = pd.read_csv('data/kemiskinan_provinsi.csv')
N = nas.set_index('periode')
nas.head(15)

## 1. Headline Maret 2026 vs Maret 2019

In [ ]:
mar19, mar26 = N.loc['Mar 2019'], N.loc['Mar 2026']
print('Persen miskin:', mar19.persen_miskin, '->', mar26.persen_miskin,
      '| delta pp:', round(mar26.persen_miskin - mar19.persen_miskin, 2))
print('Jumlah (jt):', mar19.jumlah_miskin_juta, '->', mar26.jumlah_miskin_juta)
print('Garis kemiskinan:', mar19.garis_kemiskinan_rp, '->', mar26.garis_kemiskinan_rp,
      '| tumbuh %:', round((mar26.garis_kemiskinan_rp/mar19.garis_kemiskinan_rp - 1)*100, 1))

## 2. Puncak pandemi & titik kembali di bawah prapandemi

In [ ]:
puncak = nas.loc[nas.persen_miskin.idxmax()]
print('Puncak:', puncak.periode, puncak.persen_miskin, '% /', puncak.jumlah_miskin_juta, 'jt')
urut = list(nas.periode)
setelah_sep19 = nas[nas.periode.isin(urut[2:])]
kembali = setelah_sep19[setelah_sep19.persen_miskin < 9.41].iloc[0]
print('Pertama di bawah 9,41%:', kembali.periode, kembali.persen_miskin)

## 3. Gap kota-desa

In [ ]:
piv = kd.pivot(index='periode', columns='daerah', values='persen_miskin')
piv['gap_pp'] = (piv['Perdesaan'] - piv['Perkotaan']).round(2)
piv.loc[['Mar 2019', 'Sep 2020', 'Mar 2026']]

## 4. Gini: puncak, titik terendah, dan pembalikan arah terakhir

In [ ]:
g = gini[gini.daerah=='Total'].set_index('periode').gini_ratio
print('Mar 2019:', g['Mar 2019'], '| Puncak:', g.idxmax(), g.max(),
      '| Terendah:', g.idxmin(), g.min(), '| Mar 2026:', g['Mar 2026'])
gk = gini[(gini.periode=='Mar 2026')].set_index('daerah').gini_ratio
print('Mar 2026 kota vs desa:', gk['Perkotaan'], 'vs', gk['Perdesaan'])

## 5. Provinsi Maret 2026: ekstrem, hitungan di atas nasional, pangsa Jawa

In [ ]:
p26 = prov[prov.periode=='Mar 2026']
print('Tertinggi:', p26.loc[p26.persen_miskin.idxmax(), 'provinsi'], p26.persen_miskin.max())
print('Terendah :', p26.loc[p26.persen_miskin.idxmin(), 'provinsi'], p26.persen_miskin.min())
print('Di atas nasional 8,07%:', (p26.persen_miskin > 8.07).sum(), 'dari', len(p26))
jawa = p26[p26.pulau=='Jawa'].jumlah_miskin_ribu.sum()
print('Pangsa Jawa (%):', round(jawa / (22.93*1000) * 100, 2))

## 6. Perubahan 2019 ke 2026 (32 provinsi sebanding)

In [ ]:
p19 = prov[prov.periode=='Mar 2019'].set_index('provinsi').persen_miskin
gab = p26.set_index('provinsi').persen_miskin.rename('p26').to_frame()
gab['p19'] = p19
gab = gab.drop(index=['Papua', 'Papua Barat'])  # wilayah 2019 tidak sebanding
gab['delta_pp'] = (gab.p26 - gab.p19).round(2)
print('Membaik:', (gab.delta_pp < 0).sum(), 'dari', len(gab))
gab.sort_values('delta_pp').head(5)